# 02. KoBERT 다중 감정 분류 (KOTE, 44감정)

> 이전 버전(`archive/02_kobert_baseline_OLD_토크나이저버그.ipynb`)은 AutoTokenizer 문제로 한국어가 [UNK]로 처리되어 폐기. 이 파일이 최신 학습 코드입니다.
>
> **11/1 김민서 수정본 반영**: EPOCHS 5, dropout 0.2, `pos_weight`(드문 감정 가중치, 1~5로 제한), AdamW `weight_decay=0.01`, gradient clipping 1.0. 임계값 탐색 범위 0.3~0.8로 확대.
> 박상현 3에폭 버전은 `archive/02_kobert_multilabel_train_박상현_3에폭.ipynb`에 보관.

**실행 방법**: 오른쪽 위 커널을 `kobert (Python 3.11)`로 고른 뒤, 위쪽 **Run All** 한 번이면 끝까지 실행됩니다.

- 토크나이저: SKT 공식 `KoBERTTokenizer` (transformers 4.44.2 필요)
- 손실함수: `BCEWithLogitsLoss` (Sigmoid 포함) → 감정마다 0~1 확률
- 처음엔 `TRAIN_N = 2000`으로 빠르게 확인하고, 잘 되면 `TRAIN_N = None`(4만 개 전체)으로 바꿔서 다시 실행하세요.

In [1]:
# 1. 기본 설정
import sys, time, random, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

import transformers
print("python      :", sys.executable)
print("transformers:", transformers.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device      :", device)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

project_root = Path.cwd().parent
raw_dir = project_root / "data" / "raw"
model_dir = project_root / "models"
model_dir.mkdir(exist_ok=True)
print("raw 폴더    :", raw_dir, raw_dir.exists())

c:\Users\hyun0\venvs\kobert\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python      : c:\Users\hyun0\venvs\kobert\Scripts\python.exe
transformers: 4.44.2
device      : cuda
raw 폴더    : e:\박상현 학교\2학년2학기\기계학습\10조_미니프로젝트#1_기계학습[00]\diary_emotion_project\data\raw True


In [2]:
# 2. 하이퍼파라미터 (여기만 바꾸면 됨)
TRAIN_N    = None    # None 이면 4만 개 전체 학습
VAL_N      = None    # None 이면 5천 개 전체

MAX_LENGTH = 64      # KOTE 문장은 대부분 짧음
BATCH_SIZE = 16      # 'CUDA out of memory' 나면 8로

EPOCHS     = 5
LR         = 2e-5

NUM_LABELS = 44

In [3]:
# 3. 감정 이름 (KOTE 라벨 순서)
LABELS = ['불평/불만','환영/호의','감동/감탄','지긋지긋','고마움','슬픔','화남/분노','존경','기대감','우쭐댐/무시함',
          '안타까움/실망','비장함','의심/불신','뿌듯함','편안/쾌적','신기함/관심','아껴주는','부끄러움','공포/무서움','절망',
          '한심함','역겨움/징그러움','짜증','어이없음','없음','패배/자기혐오','귀찮음','힘듦/지침','즐거움/신남','깨달음',
          '죄책감','증오/혐오','흐뭇함(귀여움/예쁨)','당황/난처','경악','부담/안 내킴','서러움','재미없음','불쌍함/연민','놀람',
          '행복','불안/걱정','기쁨','안심/신뢰']
print("레이블 수:", len(LABELS))

레이블 수: 44


In [4]:
# 4. 데이터 불러오기 (원본 tsv에서 바로 → pkl 버전 문제 없음)
def load_kote(name):
    df = pd.read_csv(raw_dir / f"{name}.tsv", sep="\t", header=None, names=["id", "text", "labels"])
    df["text"] = df["text"].astype(str)
    df["label_list"] = df["labels"].apply(lambda s: [int(x) for x in str(s).split(",")])
    df["multi_hot"] = df["label_list"].apply(lambda ls: [1 if i in ls else 0 for i in range(NUM_LABELS)])
    return df

train_df = load_kote("train")
val_df   = load_kote("val")
test_df  = load_kote("test")
print("train:", train_df.shape, "| val:", val_df.shape, "| test:", test_df.shape)
print("문장당 평균 감정 수:", round(train_df["label_list"].apply(len).mean(), 2))

if TRAIN_N: train_df = train_df.sample(n=TRAIN_N, random_state=SEED)
if VAL_N:   val_df   = val_df.sample(n=VAL_N, random_state=SEED)
print("사용할 데이터 → train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

train: (40000, 5) | val: (5000, 5) | test: (5000, 5)
문장당 평균 감정 수: 7.91
사용할 데이터 → train: 40000 | val: 5000 | test: 5000


In [5]:
# 5. KoBERT 토크나이저 + 정상 동작 확인
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel

MODEL_NAME = "skt/kobert-base-v1"
tokenizer = KoBERTTokenizer.from_pretrained(MODEL_NAME)

check = tokenizer.encode("한국어 모델을 공유합니다.")
print(tokenizer.convert_ids_to_tokens(tokenizer("오늘 너무 피곤하다")["input_ids"]))
assert check == [2, 4958, 6855, 2046, 7088, 1050, 7843, 54, 3], "토크나이저 이상! transformers 4.44.2 + kobert_tokenizer 확인"
print("토크나이저 정상 ✅")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


['[CLS]', '▁오늘', '▁너무', '▁피', '곤', '하다', '[SEP]']
토크나이저 정상 ✅


c:\Users\hyun0\venvs\kobert\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
# 6. Dataset / DataLoader
class KOTEDataset(Dataset):
    def __init__(self, df):
        self.texts  = df["text"].tolist()
        self.labels = df["multi_hot"].tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    enc["labels"] = torch.tensor(labels, dtype=torch.float32)
    return enc

train_loader = DataLoader(KOTEDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(KOTEDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(KOTEDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

b = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in b.items()})

{'input_ids': (16, 64), 'token_type_ids': (16, 64), 'attention_mask': (16, 64), 'labels': (16, 44)}


In [7]:
#** 클래스별 positive weight 계산

train_labels = np.stack(train_df["multi_hot"].values)

positive_count = train_labels.sum(axis=0)
negative_count = len(train_labels) - positive_count

pos_weight = negative_count / np.maximum(positive_count, 1)

# 너무 큰 weight 방지
pos_weight = np.clip(pos_weight, 1.0, 5.0)    # Multi-label 분류에서 적게 등장하는 감정의 정답(1)을 더 중요하게 보도록 주는 가중치

pos_weight = torch.tensor(
    pos_weight,
    dtype=torch.float32
).to(device)

print("pos_weight shape:", pos_weight.shape)
print(pos_weight)

pos_weight shape: torch.Size([44])
tensor([1.3958, 3.3234, 2.7113, 5.0000, 5.0000, 5.0000, 2.2523, 5.0000, 2.5724,
        5.0000, 1.2903, 5.0000, 2.4587, 5.0000, 5.0000, 2.7601, 4.5765, 5.0000,
        5.0000, 5.0000, 2.2784, 5.0000, 1.6792, 1.5006, 5.0000, 5.0000, 5.0000,
        5.0000, 2.6809, 3.9074, 5.0000, 4.2056, 5.0000, 3.0016, 5.0000, 5.0000,
        5.0000, 5.0000, 5.0000, 4.4385, 4.3526, 4.3043, 3.0132, 4.0448],
       device='cuda:0')


In [8]:
# 7. 모델: KoBERT → [CLS] → Linear(768→44)
class KoBERTMultiLabelClassifier(nn.Module):
    def __init__(self, num_labels=NUM_LABELS, dropout=0.2):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_vector = out.last_hidden_state[:, 0, :]          # [CLS] (KoBERTTokenizer는 맨 앞)
        return self.classifier(self.dropout(cls_vector))     # logits (B, 44)

model = KoBERTMultiLabelClassifier().to(device)
criterion = nn.BCEWithLogitsLoss(                            # Sigmoid + 이진 교차엔트로피
    pos_weight=pos_weight                                    # 상대적으로 적은 감정이 무시되는 문제를 줄이는 데 도움
)                          
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=LR,
    weight_decay=0.01                                       # 가중치가 지나치게 커지는 걸 막아서 과적합을 줄이는 역할
)
print("파라미터 수:", sum(p.numel() for p in model.parameters()) // 10**6, "M")

파라미터 수: 92 M


In [9]:
# 8. 평가 함수
def run_inputs(batch):
    return dict(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                token_type_ids=batch.get("token_type_ids"))

@torch.no_grad()
def predict_probs(loader):
    model.eval()
    P, Y = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        P.append(torch.sigmoid(model(**run_inputs(batch))).cpu().numpy())
        Y.append(batch["labels"].cpu().numpy())
    return np.concatenate(P), np.concatenate(Y)

def f1s(probs, labels, th):
    pred = (probs >= th).astype(int)
    return (f1_score(labels, pred, average="micro", zero_division=0),
            f1_score(labels, pred, average="macro", zero_division=0))

In [10]:
# 9. 학습
start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        loss = criterion(model(**run_inputs(batch)), batch["labels"])
        loss.backward()

        torch.nn.utils.clip_grad_norm_(   # 전체 학습에서는 gradient가 너무 커지는 걸 막아 안정적으로 학습하는 데 도움
            model.parameters(),
            max_norm=1.0
            )
        
        optimizer.step()
        total += loss.item()
        if step % 100 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {loss.item():.4f}")

    vp, vy = predict_probs(val_loader)
    mi, ma = f1s(vp, vy, 0.3)
    print(f"[Epoch {epoch}] train loss {total/len(train_loader):.4f} | val micro-F1 {mi:.4f} macro-F1 {ma:.4f} (th=0.3) "
          f"| {(time.time()-start)/60:.1f}분")

  epoch 1 step 100/2500 loss 0.8855
  epoch 1 step 200/2500 loss 0.7849
  epoch 1 step 300/2500 loss 0.8036
  epoch 1 step 400/2500 loss 0.7657
  epoch 1 step 500/2500 loss 0.7285
  epoch 1 step 600/2500 loss 0.7517
  epoch 1 step 700/2500 loss 0.7743
  epoch 1 step 800/2500 loss 0.7672
  epoch 1 step 900/2500 loss 0.7297
  epoch 1 step 1000/2500 loss 0.6490
  epoch 1 step 1100/2500 loss 0.8398
  epoch 1 step 1200/2500 loss 0.8037
  epoch 1 step 1300/2500 loss 0.6700
  epoch 1 step 1400/2500 loss 0.7042
  epoch 1 step 1500/2500 loss 0.7263
  epoch 1 step 1600/2500 loss 0.7143
  epoch 1 step 1700/2500 loss 0.5571
  epoch 1 step 1800/2500 loss 0.5585
  epoch 1 step 1900/2500 loss 0.6761
  epoch 1 step 2000/2500 loss 0.7183
  epoch 1 step 2100/2500 loss 0.5968
  epoch 1 step 2200/2500 loss 0.7214
  epoch 1 step 2300/2500 loss 0.7040
  epoch 1 step 2400/2500 loss 0.6401
  epoch 1 step 2500/2500 loss 0.6790
[Epoch 1] train loss 0.7532 | val micro-F1 0.4975 macro-F1 0.4490 (th=0.3) | 27.0분
 

In [12]:
# 10. 검증셋으로 임계값(threshold) 고르기
vp, vy = predict_probs(val_loader)
best_th, best_f1 = 0.5, -1
for th in [0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]:   # pos_weight 쓰면 최적값이 0.5보다 높아짐:
    mi, ma = f1s(vp, vy, th)
    print(f"th {th:.2f} | micro-F1 {mi:.4f} | macro-F1 {ma:.4f}")
    if mi > best_f1:
        best_th, best_f1 = th, mi
print("\n선택한 임계값:", best_th)

th 0.30 | micro-F1 0.5497 | macro-F1 0.4927
th 0.40 | micro-F1 0.5788 | macro-F1 0.5150
th 0.50 | micro-F1 0.5986 | macro-F1 0.5286
th 0.55 | micro-F1 0.6054 | macro-F1 0.5337
th 0.60 | micro-F1 0.6086 | macro-F1 0.5331
th 0.65 | micro-F1 0.6077 | macro-F1 0.5298
th 0.70 | micro-F1 0.6016 | macro-F1 0.5207
th 0.75 | micro-F1 0.5876 | macro-F1 0.5024
th 0.80 | micro-F1 0.5623 | macro-F1 0.4733

선택한 임계값: 0.6


In [13]:
# 11. 테스트셋 최종 성능
tp, ty = predict_probs(test_loader)
mi, ma = f1s(tp, ty, best_th)
print(f"TEST (th={best_th}) → micro-F1 {mi:.4f} | macro-F1 {ma:.4f}")

TEST (th=0.6) → micro-F1 0.6051 | macro-F1 0.5327


In [14]:
# 12. 모델 저장
save_path = model_dir / f"kobert_kote_n{TRAIN_N or 'all'}_e{EPOCHS}.pt"
torch.save({"state_dict": model.state_dict(), "threshold": best_th, "labels": LABELS,
            "max_length": MAX_LENGTH}, save_path)
print("저장:", save_path)

저장: e:\박상현 학교\2학년2학기\기계학습\10조_미니프로젝트#1_기계학습[00]\diary_emotion_project\models\kobert_kote_nall_e5.pt


In [16]:
# 13. 예측 함수 + 문장 테스트
@torch.no_grad()
def predict_text(text, top_k=5, th=None):
    th = best_th if th is None else th
    model.eval()
    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    probs = torch.sigmoid(model(**run_inputs(enc)))[0].cpu().numpy()
    order = probs.argsort()[::-1]
    picked = [(LABELS[i], float(probs[i])) for i in order if probs[i] >= th]
    top = [(LABELS[i], float(probs[i])) for i in order[:top_k]]
    return picked, top, probs

for s in ["오늘 시험을 잘 봐서 뿌듯하다.", "다음 발표가 걱정된다.", "너무 피곤하다.",
          "오늘 시험을 잘 봐서 뿌듯했지만 다음 발표가 걱정되고 너무 피곤하다."]:
    picked, top, _ = predict_text(s)
    print("\n문장:", s)
    print("  선택(≥%.2f):" % best_th, ", ".join(f"{e} {p:.2f}" for e, p in picked) or "없음")
    print("  상위 5    :", ", ".join(f"{e} {p:.2f}" for e, p in top))


문장: 오늘 시험을 잘 봐서 뿌듯하다.
  선택(≥0.60): 기쁨 0.99, 행복 0.98, 고마움 0.96, 즐거움/신남 0.95, 뿌듯함 0.83, 감동/감탄 0.82, 편안/쾌적 0.77, 환영/호의 0.74, 안심/신뢰 0.71
  상위 5    : 기쁨 0.99, 행복 0.98, 고마움 0.96, 즐거움/신남 0.95, 뿌듯함 0.83

문장: 다음 발표가 걱정된다.
  선택(≥0.60): 불안/걱정 0.92, 의심/불신 0.72, 기대감 0.68
  상위 5    : 불안/걱정 0.92, 의심/불신 0.72, 기대감 0.68, 안타까움/실망 0.59, 불평/불만 0.50

문장: 너무 피곤하다.
  선택(≥0.60): 힘듦/지침 0.94, 지긋지긋 0.77, 짜증 0.73, 불안/걱정 0.63, 귀찮음 0.61
  상위 5    : 힘듦/지침 0.94, 지긋지긋 0.77, 짜증 0.73, 불안/걱정 0.63, 귀찮음 0.61

문장: 오늘 시험을 잘 봐서 뿌듯했지만 다음 발표가 걱정되고 너무 피곤하다.
  선택(≥0.60): 힘듦/지침 0.92, 기대감 0.72, 고마움 0.65, 행복 0.62, 기쁨 0.62
  상위 5    : 힘듦/지침 0.92, 기대감 0.72, 고마움 0.65, 행복 0.62, 기쁨 0.62


In [25]:
# 14. (다음 단계 맛보기) 일기 → 문장 분리 → 하루 감정 비율
def analyze_diary(diary, top_k=5):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", diary) if s.strip()]
    all_probs = []
    for s in sents:
        picked, _, probs = predict_text(s)
        all_probs.append(probs)
        print(f"- {s}\n    → " + (", ".join(f"{e} {p:.2f}" for e, p in picked) or "선택 없음"))
    day = np.mean(all_probs, axis=0)                      # 문장별 확률 평균 = 하루 감정
    top = day.argsort()[::-1][:top_k]
    print("\n[하루 감정 Top %d]" % top_k)
    for i in top:
        print(f"  {LABELS[i]:12} {day[i]:.2f}")
    return day

_ = analyze_diary("오늘 시험을 잘 봐서 뿌듯했다. 그런데 다음 주 발표가 걱정된다. 알바까지 하고 와서 너무 피곤하다.")

- 오늘 시험을 잘 봐서 뿌듯했다.
    → 기쁨 0.98, 행복 0.98, 고마움 0.96, 즐거움/신남 0.94, 뿌듯함 0.83, 감동/감탄 0.79, 편안/쾌적 0.77, 안심/신뢰 0.71, 환영/호의 0.67
- 그런데 다음 주 발표가 걱정된다.
    → 불안/걱정 0.91, 기대감 0.81, 의심/불신 0.65
- 알바까지 하고 와서 너무 피곤하다.
    → 힘듦/지침 0.96, 지긋지긋 0.90, 짜증 0.88, 불평/불만 0.72, 귀찮음 0.70, 불쌍함/연민 0.66, 불안/걱정 0.60

[하루 감정 Top 5]
  힘듦/지침        0.53
  불안/걱정        0.52
  기대감          0.44
  지긋지긋         0.43
  짜증           0.40
